In [1]:
unique_pdb_list = [
'5YJ0',
 '5FQD',
 '1Y1Q',
 '6R0U',
 '1AF2',
 '4WPL',
 '4V30',
 '5LMO',
 '4SKN',
 '5YJ1',
 '1BO8',
 '4DW5',
 '5Y48',
 '1XMY',
 '1OYN',
 '4WRV',
 '7BQU',
 '8OU3',
 '1UDH',
 '1LNX',
 '3BJE',
 '1M8V',
 '1WS3',
 '8AOQ',
 '3QE7',
 '5WAN',
 '2V0L',
 '1Q9M',
 '3PNS',
 '9DOM',
 '8BC7',
 '4JYK',
 '2EUG',
 '1KSL',
 '6PAI',
 '6UGJ',
 '2HBM',
 '1TBB',
 '6R12',
 '9H59',
 '5S46',
 '1SSP',
 '4DW7',
 '5LHV',
 '5V3O',
 '1GTH',
 '3WX2',
 '6AJO',
 '6KNL',
 '1TLZ',
 '6Q0R',
 '6SGL',
 '1BD4',
 '4TXL',
 '3IVD',
 '4TZC',
 '8OU7',
 '3QPB',
 '4TZ4',
 '4R07',
 '7Q2W',
 '7PSO',
 '1Q3F',
 '7SZB',
 '8OU5',
 '8DEY',
 '7M7K',
 '4KPV',
 '5YIZ',
 '5OH4',
 '6R1D',
 '2FR6',
 '4X0F',
 '1I5L',
 '6IOA',
 '5XLS',
 '1BO7',
 '4HK7',
 '4PD6',
 '4CI2',
 '5ILW',
 '1UAQ',
 '1KSV',
 '1XN0',
 '1UI0',
 '5OH7',
 '4V32',
 '6YAB',
 '3SLF',
 '3CXM',
 '5OO4',
 '3I0F',
 '8TZX',
 '3G1D',
 '2XRF',
 '1LOQ',
 '5ILX',
 '4EG2',
 '6TX9',
 '5OO8',
 '1Y1S',
 '8G66',
 '1TGY',
 '6IOB',
 '8OU9',
 '3G4K',
 '5AMI',
 '4MML',
 '4WRU',
 '4ZBY',
 '6TFL',
 '6TEG',
 '5OOA',
 '2RJ3',
 '6R1W',
 '4OEH',
 '4YYY',
 '6K5H',
 '5L26',
 '9DJX',
 '5M2T',
 '3KVY',
 '5AMJ',
 '5GNW',
 '3G22',
 '6R11',
 '1RO6',
 '1OE5',
 '5OH1',
 '4V2Y',
 '1KLZ',
 '4R09',
 '8TNR',
 '6R1X',
 '6R1C',
 '7M32',
 '6R19',
 '4LZB',
 '1EMJ',
 '5OO9',
 '8OUA',
 '5AMH',
 '6IOC',
 '8TNQ',
 '4R0A',
 '1JLS',
 '7U8F',
 '8AOP',
 '7QSH',
 '4CI1',
 '5C80',
 '4YIG',
 '6R1A',
 '8OU4',
 '5AMK',
 '5OH8',
 '7LPS',
 '1UCD',
 '6R18',
 '1KM4',
 '6XK9',
 '1T36',
 '2HWU',
 '1BRW',
 '2FVK',
 '4R2W',
 '7VTE',
 '6R13',
 '1LOJ',
 '4YJK',
 '6R0S',
 '8OU6',
 '6R1K',
 '4YGM',
 '1A34',
 '6AJR',
 '3LOC',
 '3TIJ',
 '7QTA',
 '7PS9',
 '3G1X',
 '3ERC',
 '1FLZ',
 '3CL7',
 '1KSK',
 '8OJH',
 '5HXB',
 '4LCS',
 '2YLC',
 '4JX9',
 '4R08',
 '5MIW',
 '8D80',
 '6Z1B',
 '5EUG',
 '6R0V',
 '4XK4',
 '7SZA']

In [2]:
import requests
from collections import Counter

def get_most_frequent_uniprot(pdb_ids):
    url = "https://data.rcsb.org/graphql"
    query = """
    query ($ids: [String!]!) {
      entries(entry_ids: $ids) {
        rcsb_id
        polymer_entities {
          rcsb_polymer_entity_container_identifiers {
            reference_sequence_identifiers {
              database_accession
              database_name
            }
          }
        }
      }
    }
    """
    
    variables = {"ids": [pdb.upper() for pdb in pdb_ids]}
    response = requests.post(url, json={'query': query, 'variables': variables})
    
    if response.status_code != 200:
        return None, None

    data = response.json()['data']['entries']
    
    # List to store every UniProt ID found (allowing for counting)
    all_found_uniprots = []
    pdb_mapping = {}

    for entry in data:
        pdb_id = entry['rcsb_id']
        current_pdb_uniprots = set() # Use a set to avoid double-counting within one PDB
        
        for entity in entry['polymer_entities']:
            identifiers = entity['rcsb_polymer_entity_container_identifiers']['reference_sequence_identifiers']
            if identifiers:
                for ref in identifiers:
                    if ref['database_name'] == 'UniProt':
                        current_pdb_uniprots.add(ref['database_accession'])
        
        all_found_uniprots.extend(list(current_pdb_uniprots))
        pdb_mapping[pdb_id] = current_pdb_uniprots

    # Tally the results
    counts = Counter(all_found_uniprots)
    
    # Sort
    counts = dict(sorted(counts.items(), key=lambda item: item[1], reverse=True))
    
    return counts, pdb_mapping

all_counts, pdb_mapping = get_most_frequent_uniprot(unique_pdb_list)

In [3]:
for pdb, ids in pdb_mapping.items():
    print(f"{pdb:<10} | {', '.join(ids) if ids else 'No UniProt found'}")

5YJ0       | Q8C7D2
5FQD       | P48729, Q16531, Q96SW2
1Y1Q       | P0A1F6
1AF2       | P0ABF6
4WPL       | P9WFQ9
4V30       | A4TVL0
5LMO       | P0DOY6, Q5SHN3, P80376, Q5SIH3, P0DOY9, P80372, P80373, P0DOY7, Q5SKU2, Q5SLQ0, Q5SHN7, P80371, P80374, Q5SHP2, P80380, P80377, P17291, Q5SHQ5, Q5SLP8, Q5SJH3, Q5SJ76, Q5SHR1
4SKN       | P13051
5YJ1       | Q8C7D2
1BO8       | P00469
4DW5       | Q96662
1XMY       | Q07343
1OYN       | Q08499
4WRV       | P9WFQ9
7BQU       | Q9UJQ4, Q96SW2
1UDH       | P10186
1LNX       | Q8ZYG5
3BJE       | Q57VZ2
1M8V       | Q9V0Y8
8AOQ       | A4TVL0
5WAN       | P75898
1Q9M       | Q08499
9DOM       | Q96SW2, Q9UKS7
8BC7       | A4TVL0
4JYK       | P0ACU2
2EUG       | P12295
1KSL       | P0AA43
6PAI       | Q16531, Q66K64, Q9BW61, Q14498
2HBM       | Q12149
1TBB       | Q08499
6R12       | A4TVL0
9H59       | Q16531, Q8TDX7, Q96SW2
5S46       | P0DTD1
1SSP       | P13051
4DW7       | Q96662
5LHV       | Q9K4U1
5V3O       | Q16531, Q96SW2
1GTH       |

In [4]:
for uid, count in all_counts.items():
    print(f"- {uid}: {count} structures")

- A4TVL0: 37 structures
- Q16531: 20 structures
- Q96SW2: 18 structures
- O26232: 6 structures
- Q8C7D2: 5 structures
- Q9K4U1: 5 structures
- P0A1F6: 4 structures
- P13051: 4 structures
- Q07343: 4 structures
- Q08499: 4 structures
- P0ACU2: 4 structures
- Q9NR97: 4 structures
- D9J2T9: 4 structures
- E2EKP5: 4 structures
- P9WFQ9: 3 structures
- P75898: 3 structures
- Q9UKS7: 3 structures
- P12295: 3 structures
- P0AA43: 3 structures
- I7F541: 3 structures
- Q13422: 3 structures
- Q8E9X9: 3 structures
- P48729: 2 structures
- P00469: 2 structures
- Q96662: 2 structures
- Q66K64: 2 structures
- Q9BW61: 2 structures
- Q14498: 2 structures
- Q28943: 2 structures
- A0QP43: 2 structures
- Q26998: 2 structures
- Q9Y3C4: 2 structures
- Q96S44: 2 structures
- Q9KPL5: 2 structures
- P0CF65: 2 structures
- O95785: 2 structures
- P0AEX9: 2 structures
- P20995: 2 structures
- P20536: 2 structures
- P15170: 2 structures
- P0ABF6: 1 structures
- P0DOY6: 1 structures
- Q5SHN3: 1 structures
- P80376

In [5]:
#get each pdb list containing uniprot ids A4TVL0 and Q96SW2 respectively (Q16531 is DDB1)
final_dict = {}
for uniportt_id in ['A4TVL0', 'Q96SW2']:
    current_list = []
    for pdb, ids in pdb_mapping.items():
        for id in ids:
            if id == uniportt_id:
                current_list.append(pdb)
                break
    final_dict[uniportt_id] = current_list

print(final_dict)

{'A4TVL0': ['4V30', '8AOQ', '8BC7', '6R12', '8OU7', '7PSO', '8OU5', '5OH4', '5OH7', '4V32', '8OU9', '5AMI', '6R1W', '5AMJ', '6R11', '5OH1', '4V2Y', '6R1X', '6R1C', '8OUA', '5AMH', '8AOP', '6R1A', '8OU4', '5AMK', '5OH8', '6R18', '6R13', '6R0S', '8OU6', '6R1K', '7PS9', '6R0V', '8OU3', '6R1D', '6R19', '6R0U'], 'Q96SW2': ['5FQD', '7BQU', '9DOM', '9H59', '5V3O', '4TZ4', '8DEY', '8TZX', '8G66', '9DJX', '8TNR', '8TNQ', '7U8F', '7LPS', '6XK9', '8OJH', '5HXB', '8D80']}
